Ce code permet de split des articles entiers en plusieurs phrases séparées. Il faut en entrée un fichier xlsx avec comme colonne :
- id l'identifiant du texte (string, clef primaire)
- text_fr le texte en français (string)
- text_eng le texte français traduit en anglais (utilisation de Deepl pour la traduction ici) (string)

L'objectif est de créer tsv avec comme colonnes :
- id l'identifiant du texte (string)
- id_sentence l'identifiant de la phrase au sein de l'article (entier)
- sentence la phrase récupérée du fichier excel (string)
Clef primaire -> (id, id_sentence)

Les articles récupérés sont disponibles sur : https://github.com/obs-info/obsinfox/blob/main/obsinfox.csv 

In [4]:
# imports
import re
import pandas as pd

In [11]:
# fonction la plus importante : permet de séparer un texte en une liste de phrase
def decomposer_en_phrases(texte):
    texte = re.sub(r'\n-\s*', '\n', texte) 
    texte = re.sub(r'(\n)', ' ', texte)  
    pattern = r'(.*?[.!?])(?:\s+|$)' 
    phrases = re.findall(pattern, texte, re.DOTALL)
    return [phrase.strip() for phrase in phrases if phrase.strip()]


# test
texte = """Phrase 1. Phrase 2 pour tester une question ? et une exclamation !
Et la j'ai sauté à la ligne."""

phrases = decomposer_en_phrases(texte)
print(phrases)


['Phrase 1.', 'Phrase 2 pour tester une question ?', 'et une exclamation !', "Et la j'ai sauté à la ligne."]


In [ ]:
# creation et remplissage du fichier tsv

input_excel = r'obsinfox articles.xlsx'  # fichier excel en entrée, a specifier
output_tsv = r'test.tsv'  # fichier tsv en sortie, a specifier

def creer_fichier_tsv(input_excel, output_tsv):
    df = pd.read_excel(input_excel)
    lignes = []
    
    for index, row in df.iterrows():
        id_text = row['id'] 
        texte = row['text_eng']

        phrases = decomposer_en_phrases(texte)
        for id_phrase, phrase in enumerate(phrases):
            lignes.append({'id_text': id_text, 'id_sentence': id_phrase, 'sentence': phrase})
    

    df_output = pd.DataFrame(lignes)

    df_output.to_csv(output_tsv, sep='\t', index=False)

creer_fichier_tsv(input_excel, output_tsv)